# gold_driver_performance

**Sources:**
- fact:      `silver.driver_shifts`  (shift_id, driver_id, earnings, rating, distance)
- dimension: `silver.drivers`        (driver_id → demographics)
- cross-ref: `silver.orders`         (driver_key → order count)

**Joins:**
- `driver_shifts.driver_id = drivers.driver_id`  (both INT)
- `orders.driver_key.cast(INT) = driver_shifts.driver_id`  (driver_key is STRING of INT)

**Grain:** one row per driver_id  
**Merge key:** `driver_id`  
**Lineage:** Silver → Gold (never Bronze directly)

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "ubereats_dev")

In [ ]:
catalog           = dbutils.widgets.get("catalog")
gold_table        = f"{catalog}.gold.driver_performance"
silver_shifts     = f"{catalog}.silver.driver_shifts"
silver_drivers    = f"{catalog}.silver.drivers"
silver_orders     = f"{catalog}.silver.orders"

print(f"[gold] {gold_table}  ←  {silver_shifts} × {silver_drivers} × {silver_orders}")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold_table} (
        driver_id              INT NOT NULL,
        first_name             STRING,
        last_name              STRING,
        city                   STRING,
        vehicle_type           STRING,
        total_shifts           LONG,
        total_orders           LONG,
        order_count            LONG,
        total_earnings_brl     DOUBLE,
        avg_shift_rating       DOUBLE,
        total_distance_km      DOUBLE,
        avg_shift_duration_min DOUBLE,
        _computed_at           TIMESTAMP NOT NULL
    ) USING DELTA
    CLUSTER BY (driver_id)
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f"[gold] table {gold_table} ready")

In [ ]:
from pyspark.sql.functions import col, count, sum, avg, countDistinct, current_timestamp

driver_shifts = spark.table(silver_shifts)
drivers       = spark.table(silver_drivers)
orders        = spark.table(silver_orders)

# Aggregate shift metrics per driver_id
# fact: silver.driver_shifts
shifts_agg = (
    driver_shifts.groupBy("driver_id")
    .agg(
        count("shift_id").alias("total_shifts"),
        sum("num_orders").alias("total_orders"),
        sum("earnings_brl").alias("total_earnings_brl"),
        avg("shift_rating").alias("avg_shift_rating"),
        sum("distance_covered_km").alias("total_distance_km"),
        avg("shift_duration_min").alias("avg_shift_duration_min"),
    )
)

# Cross-ref: order_count per driver from silver.orders
# orders.driver_key is STRING containing the integer driver_id value
driver_order_counts = (
    orders
    .filter(col("driver_key").isNotNull())
    .withColumn("driver_id_int", col("driver_key").cast("integer"))
    .groupBy("driver_id_int")
    .agg(countDistinct("order_id").alias("order_count"))
)

# Enrich with driver demographics and cross-ref order count
# dimension: silver.drivers → driver_id, first_name, last_name, city, vehicle_type
perf_df = (
    shifts_agg
    .join(
        drivers.select("driver_id", "first_name", "last_name", "city", "vehicle_type"),
        on="driver_id",
        how="left",
    )
    .join(
        driver_order_counts,
        shifts_agg["driver_id"] == driver_order_counts["driver_id_int"],
        how="left",
    )
    .select(
        shifts_agg["driver_id"],
        col("first_name"),
        col("last_name"),
        col("city"),
        col("vehicle_type"),
        col("total_shifts"),
        col("total_orders"),
        col("order_count"),
        col("total_earnings_brl"),
        col("avg_shift_rating"),
        col("total_distance_km"),
        col("avg_shift_duration_min"),
    )
    .withColumn("_computed_at", current_timestamp())
)

print(f"[gold] {perf_df.count()} drivers")

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, desc, col

# silver.drivers is deduped/merged by uuid, not driver_id (DESIGN_GOLD_DIMENSION_JOIN_INTEGRITY.md).
# The new check=unique rule on driver_id (contracts/drivers.yml) should keep this from
# happening going forward, but this guard stays as defense-in-depth for any rows that
# landed before the rule existed.
w = Window.partitionBy("driver_id").orderBy(desc("_computed_at"))
perf_df_deduped = (
    perf_df
    .withColumn("_rn", row_number().over(w))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

perf_df_deduped.createOrReplaceTempView("gold_driver_performance_batch")

spark.sql(f"""
    MERGE INTO {gold_table} AS t
    USING gold_driver_performance_batch AS s
    ON t.driver_id = s.driver_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"[gold] {gold_table} updated")